In [1]:
import sympy as sp

In [ ]:
def identity_matrix(n):
    """Tạo ma trận đơn vị cỡ n x n"""
    return [[sp.Integer(1) if i == j else sp.Integer(0) for j in range(n)] for i in range(n)]

In [2]:
def copy_matrix(M):
    """Sao chép sâu một ma trận list"""
    return [[cell for cell in row] for row in M]

In [3]:
def transpose(M):
    """Chuyển vị ma trận"""
    return [[M[j][i] for j in range(len(M))] for i in range(len(M[0]))]

In [4]:
def get_submatrix(M, row_to_remove, col_to_remove):
    """Lấy ma trận con bằng cách bỏ đi một dòng và một cột"""
    return [[M[i][j] for j in range(len(M[i])) if j != col_to_remove] 
            for i in range(len(M)) if i != row_to_remove]

In [ ]:
def solve_linear(A):
    return None

In [ ]:
def solve_quadractic(A):
    return None

In [9]:
def determinant(M):
    """Tính định thức của ma trận vuông bằng khai triển Laplace"""
    n = len(M)
    if n == 1:
        return M[0][0]
    if n == 2:
        return M[0][0] * M[1][1] - M[0][1] * M[1][0]
    
    det = sp.Integer(0)
    for col in range(n):
        sign = (-1) ** col
        sub = get_submatrix(M, 0, col)
        det += sign * M[0][col] * determinant(sub)
    return sp.expand(det)

In [ ]:
def rref(M_augmented):
    """
    Đưa ma trận bổ sụng về dạng bậc thang rút gọn (RREF) bằng biến đổi Gauss.
    M_augmented là list dạng biến số của Sympy.
    """
    A = copy_matrix(M_augmented)
    rows = len(A)
    cols = len(A[0])
    
    pivot_row = 0
    for col in range(cols - 1): # Không xét cột hệ số tự do cuối cùng nếu có, ở đây giải hệ thuần nhất
        # Tìm dòng có phần tử khác 0 để làm pivot
        swap_row = -1
        for r in range(pivot_row, rows):
            if sp.simplify(A[r][col]) != 0:
                swap_row = r
                break
                
        if swap_row == -1:
            continue
            
        A[pivot_row], A[swap_row] = A[swap_row], A[pivot_row]
        
        # Chuẩn hóa dòng pivot về 1
        pivot_val = A[pivot_row][col]
        A[pivot_row] = [sp.simplify(x / pivot_val) for x in A[pivot_row]]
        
        # Khử các dòng khác
        for r in range(rows):
            if r != pivot_row:
                factor = A[r][col]
                A[r] = [sp.simplify(item_r - factor * item_p) for item_r, item_p in zip(A[r], A[pivot_row])]
        
        pivot_row += 1
        if pivot_row >= rows:
            break
    return A

In [ ]:
def characteristic_polynomial(M):
    lam = sp.Symbol('lamda')
    for i in range(len(M)):
        M[i][i] -= lam
    
    return determinant(M) 

In [ ]:
def polynomial_coeff(poly):
    lamda = sp.Symbol('lamda')
    P = sp.Poly(sp.expand(poly), lamda)
    return [float(c) for c in P.all_coeffs()]

In [ ]:
def solve_poly(C):
    if len(C) == 0 or len(C) == 1:
        print("No solution")
    elif len(C) == 2:
        return solve_linear()
    elif len(C) == 3:
        return solve_quadractic()

    

In [ ]:
def find_eigenvectors(A, lam, mult):
    """Tìm các vectơ riêng độc lập tuyến tính ứng với giá trị riêng lam"""
    n = len(A)
    # Tạo ma trận (A - lam * I)
    E = copy_matrix(A)
    for i in range(n):
        E[i][i] = sp.simplify(E[i][i] - lam)
        
    # Đưa về dạng bậc thang rút gọn RREF
    E_rref = rref(E)
    
    # Xác định các cột tự do (free variables)
    pivot_cols = []
    for r in range(n):
        for c in range(n):
            if sp.simplify(E_rref[r][c]) == 1:
                pivot_cols.append(c)
                break
                
    free_cols = [c for c in range(n) if c not in pivot_cols]
    
    eigenvectors = []
    # Tìm hệ nghiệm cơ bản từ các biến tự do
    for free in free_cols:
        vec = [sp.Integer(0)] * n
        vec[free] = sp.Integer(1)
        for r in range(len(pivot_cols)):
            p_col = pivot_cols[r]
            if p_col < free: # Chỉ lấy các giá trị ảnh hưởng bởi biến tự do này
                vec[p_col] = sp.simplify(-E_rref[r][free])
        eigenvectors.append(vec)
        
    return eigenvectors

In [ ]:
def diagonalize(A):
    n = len(A)
    lam = sp.Symbol('lam')
    
    # Tạo ma trận A - lam*I
    A_lam = copy_matrix(A)
    for i in range(n):
        A_lam[i][i] = A_lam[i][i] - lam
        
    # Bước 1: Tính đa thức đặc trưng và giải tìm giá trị riêng
    char_poly = determinant(A_lam)
    print(f"Đa thức đặc trưng: {char_poly} = 0")
    
    eigenvals_dict = sp.solve(char_poly, lam)
    if not eigenvals_dict:
        raise ValueError("Không tìm thấy giá trị riêng thực/phức phù hợp.")
        
    # Tính bội đại số của từng nghiệm
    # sp.solve trả về list các nghiệm, ta gom nhóm để biết số lần xuất hiện (bội)
    eigenvalues = []
    for val in set(eigenvals_dict):
        # Tính bội bằng cách đếm số lần nghiệm xuất hiện hoặc kiểm tra nghiệm bội
        # Đơn giản nhất là kiểm tra đạo hàm hoặc dùng trực tiếp từ sp.roots nếu muốn, 
        # nhưng ở đây ta đếm số lượng xuất hiện từ sp.solve quy đổi gốc
        count = eigenvals_dict.count(val)
        eigenvalues.append((val, count))
    
    P_cols = []
    D_elements = []
    
    # Bước 2: Tìm vectơ riêng cho từng giá trị riêng
    for val, mult in eigenvalues:
        vecs = find_eigenvectors(A, val, mult)
        
        # Nếu số lượng vectơ riêng (bội hình học) < bội đại số -> Không chéo hóa được
        if len(vecs) < mult:
            print(f"Giá trị riêng {val} có bội đại số {mult} nhưng chỉ có {len(vecs)} vectơ riêng.")
            return None, None
            
        for v in vecs:
            P_cols.append(v)
            D_elements.append(val)
            
    if len(P_cols) < n:
        return None, None
        
    # Tạo ma trận P (chuyển vị của P_cols vì P_cols đang chứa các vectơ cột)
    P = transpose(P_cols)
    
    # Tạo ma trận đường chéo D
    D = [[D_elements[i] if i == j else sp.Integer(0) for j in range(n)] for i in range(n)]
    
    return P, D

In [ ]:
if __name__ == "__main__":
    # Khai báo ma trận bằng cấu trúc list lồng nhau
    # Ví dụ ma trận A cỡ 3x3
    A = [
        [sp.Integer(4), sp.Integer(1), sp.Integer(-1)],
        [sp.Integer(2), sp.Integer(5), sp.Integer(-2)],
        [sp.Integer(1), sp.Integer(1), sp.Integer(2)]
    ]
    
    print(characteristic_polynomial(A))

-lamda**3 + 11*lamda**2 - 39*lamda + 45
